In [ ]:
%env PYTHONHASHSEED=0
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.pyplot import rc_context
import seaborn as sns
import scanpy as sc
import anndata as ad
import rpy2

In [ ]:
np.random.seed(0)
sc.set_figure_params(dpi = 300, dpi_save = 300, frameon = False)

In [ ]:
%load_ext rpy2.ipython

In [ ]:
%%R

library(SingleCellExperiment)
library(dplyr)
library(scater)
library(BiocParallel)
library(scDblFinder)

In [ ]:
# import samples count matrices from CellBender output files and generate anndata object
samples = {
    "CART_3": "../output/cellbender_outputs_v2/human/cellbender_outputs/CAR_T_3/CAR_T_3_output_adjusted_filtered.h5",
    "CART_8": "../output/cellbender_outputs_v2/human/cellbender_outputs/CAR_T_8/CAR_T_8_output_adjusted_filtered.h5",
    "CART_9": "../output/cellbender_outputs_v2/human/cellbender_outputs/CAR_T_9/CAR_T_9_output_adjusted_filtered.h5",
    "CART_10": "../output/cellbender_outputs_v2/human/cellbender_outputs/CAR_T_10/CAR_T_10_output_adjusted_filtered.h5",
    "DIPG_73": "../output/cellbender_outputs_v2/human/cellbender_outputs/DIPG_73/DIPG_73_output_adjusted_filtered.h5",
    "DIPG_79": "../output/cellbender_outputs_v2/human/cellbender_outputs/DIPG_79_Cortex/DIPG_79_Cortex_output_adjusted_filtered.h5",
    "DIPG_95": "../output/cellbender_outputs_v2/human/cellbender_outputs/DIPG_95_Cortex/DIPG_95_Cortex_output_adjusted_filtered.h5",
}

adatas = {}

for sample_id, path in samples.items():
    sample_adata = sc.read_10x_h5(path)
    sample_adata.var_names_make_unique()
    adatas[sample_id] = sample_adata

adata = ad.concat(adatas, label="sample")
adata.obs_names_make_unique()

In [ ]:
# add metadata
condition_key = {
    "CART_3": "CAR T",
    "CART_8": "CAR T",
    "CART_9": "CAR T",
    "CART_10": "CAR T",
    "DIPG_73": "Control",
    "DIPG_79": "Control",
    "DIPG_95": "Control",
}

batch_key = {
    "CART_3": "A",
    "CART_8": "B",
    "CART_9": "A",
    "CART_10": "B",
    "DIPG_73": "A",
    "DIPG_79": "B",
    "DIPG_95": "B",
}

age_key = {
    "CART_3": 22,
    "CART_8": 13,
    "CART_9": 5,
    "CART_10": 12,
    "DIPG_73": 19,
    "DIPG_79": 13,
    "DIPG_95": 13,
}

sex_key = {
    "CART_3": "Male",
    "CART_8": "Female",
    "CART_9": "Female",
    "CART_10": "Male",
    "DIPG_73": "Male",
    "DIPG_79": "Male",
    "DIPG_95": "Female",
}

sampleid_key = {
    "CART_3": "CART_A",
    "CART_8": "CART_B",
    "CART_9": "CART_C",
    "CART_10": "CART_D",
    "DIPG_73": "Control_A",
    "DIPG_79": "Control_B",
    "DIPG_95": "Control_C",
}

adata.obs['condition'] = adata.obs['sample'].map(condition_key)
adata.obs['batch'] = adata.obs['sample'].map(batch_key)
adata.obs['age'] = adata.obs['sample'].map(age_key)
adata.obs['sex'] = adata.obs['sample'].map(sex_key)
adata.obs['sampleid'] = adata.obs['sample'].map(sampleid_key)

adata

In [ ]:
# first-pass filtering of cells with very low gene and UMI counts 
sc.pp.filter_cells(adata, min_genes=100)
sc.pp.filter_cells(adata, min_counts=200)

In [ ]:
adata

In [ ]:
# add feature metadata denoting mitochondrial, ribosomal, or hemoglobin genes for QC metrics

# mitochondrial genes
adata.var["mt"] = adata.var_names.str.startswith("MT-")
# ribosomal genes
adata.var["ribo"] = adata.var_names.str.startswith(("RPS", "RPL"))
# hemoglobin genes
adata.var["hb"] = adata.var_names.str.contains("^HB[^(P)|^(EGF)|^(S1L)]")

In [ ]:
# calculate qc metrics
sc.pp.calculate_qc_metrics(
    adata, qc_vars=["mt", "ribo", "hb"], inplace=True, log1p=True
)

In [ ]:
# plotting QC metrics
with rc_context({"figure.figsize": (5, 3), "grid.alpha":0}):
    sc.pl.violin(
        adata,
        ["n_genes_by_counts", "total_counts", "pct_counts_mt"],
        jitter=0.4,
        size=0.3,
        multi_panel=True
    )

In [ ]:
with rc_context({"figure.figsize": (7.5, 3), "grid.alpha":0}):
    # plot QC metric
    sc.pl.violin(
        adata,
        ["n_genes_by_counts"],
        groupby = 'sampleid',
        jitter=0.4,
        size=0.3,
        order=['Control_A','Control_B','Control_C', 'CART_A','CART_B','CART_C','CART_D'],
        palette={"Control_A":"#0e427c", "Control_B":"#5a97c1", "Control_C":"#74c8c3",
                 "CART_A":"#6f1806", "CART_B":"#dd2c09", "CART_C":"#e67424", "CART_D":"#f5c34d"},
        ylabel='Unique genes expressed',
        xlabel=None
    )

In [ ]:
with rc_context({"figure.figsize": (7.5, 3), "grid.alpha":0}):
    sc.pl.violin(
        adata,
        ["log1p_n_genes_by_counts"],
        groupby = 'sampleid',
        jitter=0.4,
        size=0.3,
        order=['Control_A','Control_B','Control_C', 'CART_A','CART_B','CART_C','CART_D'],
        palette={"Control_A":"#0e427c", "Control_B":"#5a97c1", "Control_C":"#74c8c3",
                 "CART_A":"#6f1806", "CART_B":"#dd2c09", "CART_C":"#e67424", "CART_D":"#f5c34d"},
        ylabel='log1p(Unique genes expressed)',
        xlabel=None
    )

In [ ]:
with rc_context({"figure.figsize": (7.5, 3), "grid.alpha":0}):
    sc.pl.violin(
        adata,
        ["total_counts"],
        groupby = 'sampleid',
        jitter=0.4,
        size=0.3,
        order=['Control_A','Control_B','Control_C', 'CART_A','CART_B','CART_C','CART_D'],
        palette={"Control_A":"#0e427c", "Control_B":"#5a97c1", "Control_C":"#74c8c3",
                 "CART_A":"#6f1806", "CART_B":"#dd2c09", "CART_C":"#e67424", "CART_D":"#f5c34d"},
        ylabel='UMIs',
        xlabel=None
    )

In [ ]:
with rc_context({"figure.figsize": (7.5, 3), "grid.alpha":0}):
    sc.pl.violin(
        adata,
        ["log1p_total_counts"],
        groupby = 'sampleid',
        jitter=0.4,
        size=0.3,
        order=['Control_A','Control_B','Control_C', 'CART_A','CART_B','CART_C','CART_D'],
        palette={"Control_A":"#0e427c", "Control_B":"#5a97c1", "Control_C":"#74c8c3",
                 "CART_A":"#6f1806", "CART_B":"#dd2c09", "CART_C":"#e67424", "CART_D":"#f5c34d"},
        ylabel='log1p(UMIs)',
        xlabel=None
    )

In [ ]:
with rc_context({"figure.figsize": (7.5, 3), "grid.alpha":0}):
    sc.pl.violin(
        adata,
        ["pct_counts_mt"],
        groupby = 'sampleid',
        jitter=0.4,
        size=0.3,
        order=['Control_A','Control_B','Control_C', 'CART_A','CART_B','CART_C','CART_D'],
        palette={"Control_A":"#0e427c", "Control_B":"#5a97c1", "Control_C":"#74c8c3",
                 "CART_A":"#6f1806", "CART_B":"#dd2c09", "CART_C":"#e67424", "CART_D":"#f5c34d"},
        ylabel='% mitochondrial UMIs',
        xlabel=None
    )

In [ ]:
with rc_context({"figure.figsize": (5, 4), "grid.alpha":0}):
    sc.pl.scatter(adata, "log1p_total_counts", "log1p_n_genes_by_counts", color="pct_counts_mt",
                 title='')

In [ ]:
with rc_context({"figure.figsize": (5, 4), "grid.alpha":0}):
    sc.pl.scatter(adata, "log1p_total_counts", "log1p_n_genes_by_counts", color="sampleid",
                 title='')

In [ ]:
# check cell counts by metadata categories
adata.obs['sampleid'].value_counts()

In [ ]:
adata.obs['condition'].value_counts()

In [ ]:
pd.crosstab(adata.obs['condition'], adata.obs['sampleid'])

In [ ]:
pd.crosstab(adata.obs['condition'], adata.obs['batch'])

In [ ]:
pd.crosstab(adata.obs['condition'], adata.obs['sex'])

In [ ]:
pd.crosstab(adata.obs['condition'], adata.obs['age'])

In [ ]:
# save concatenated anndata object as h5ad file
adata.write('../output/human/human_concatenated_input_object.h5ad')

In [ ]:
# extract count matrix, and cell/feature metadata for generating a SingleCellExperiment object in R
count_matrix = adata.X
count_matrix = count_matrix.todense().transpose()
sce_rowdata = adata.var
sce_coldata = adata.obs

In [ ]:
%%R -i count_matrix -i sce_rowdata -i sce_coldata
# create SCE object
sce <- SingleCellExperiment(list(counts = count_matrix),
                              rowData = sce_rowdata,
                              colData = sce_coldata)

In [ ]:
%%R

sce

In [ ]:
%%R

saveRDS(sce, '../output/human/human_prefiltering_sce_object.rds')

In [ ]:
%%R
# run scDblFinder doublet algorithm
sce <- scDblFinder(sce, samples="sampleid", BPPARAM=MulticoreParam(36))

In [ ]:
%%R
# check the number of doublets called per sample
table(sce$scDblFinder.class, sce$sampleid)

In [ ]:
%%R

# filter outlier cells based on number of UMIs, number of genes detected, and % mitochondrial UMIs
cols <- c("total_counts", "n_genes_by_counts", "pct_counts_mt")
log <- c(TRUE, TRUE, FALSE)
type <- c("both", "both", "higher")
nmad <- c(3, 3, 3)
nmad2 <- c(3, 3, 3)

drop_cols <- paste0(cols, "_drop")
drop_cols2 <- paste0(cols, "_drop2")

for (i in seq_along(cols)){
  colData(sce)[[drop_cols[i]]] <- isOutlier(sce[[cols[i]]], nmads = nmad[i], type = type[i], log = log[i])
    colData(sce)[[drop_cols2[i]]] <- isOutlier(sce[[cols[i]]], nmads = nmad2[i], type = type[i], log = log[i], batch = sce$sampleid)
}

In [ ]:
%%R

drop_cols3 <- c(drop_cols, drop_cols2)

# check the proportion of cells filtered from each sample following outlier and doublet filtering
ol <- rowSums(as.matrix(colData(sce)[drop_cols3])) != 0
dubs <- sce$scDblFinder.class == "doublet"

# summary of cells kept
ns <- table(sce$sample)
ns_fil <- table(sce$sampleid[!ol])
ns_nondub <- table(sce$sampleid[!dubs])

ns_nondub_fil <- table(sce$sampleid[!((ol + dubs) != 0)])

print(rbind(
  Unfiltered = round(ns, digits = 0), Filtered = round(ns_fil, digits = 0), 
    Singlets = round(ns_nondub, digits = 0), Singlets_Filtered = round(ns_nondub_fil, digits = 0), 
  "% passing QC" = paste0(round(ns_nondub_fil / ns * 100, digits = 0), "%")), quote = FALSE)

In [ ]:
%%R
# add QC filtering info to SCE metadata
sce$passed_qc <- !((ol + dubs) != 0)

In [ ]:
%%R -o qc_outcomes
# export this metadata to python
qc_outcomes <- sce$passed_qc

In [ ]:
# add metadata to python object
adata.obs['passed_qc'] = qc_outcomes

In [ ]:
# plot QC metrics again following filtering
with rc_context({"figure.figsize": (5, 3), "grid.alpha":0}):
    sc.pl.violin(
        adata[adata.obs['passed_qc'] == 1],
        ["n_genes_by_counts", "total_counts", "pct_counts_mt"],
        jitter=0.4,
        size=0.3,
        multi_panel=True
    )

In [ ]:
with rc_context({"figure.figsize": (7.5, 3), "grid.alpha":0}):
    sc.pl.violin(
        adata[adata.obs['passed_qc'] == 1],
        ["n_genes_by_counts"],
        groupby = 'sampleid',
        jitter=0.4,
        size=0.3,
        order=['Control_A','Control_B','Control_C', 'CART_A','CART_B','CART_C','CART_D'],
        palette={"Control_A":"#0e427c", "Control_B":"#5a97c1", "Control_C":"#74c8c3",
                 "CART_A":"#6f1806", "CART_B":"#dd2c09", "CART_C":"#e67424", "CART_D":"#f5c34d"},
        ylabel='Unique genes expressed',
        xlabel=None
    )

In [ ]:
with rc_context({"figure.figsize": (7.5, 3), "grid.alpha":0}):
    sc.pl.violin(
        adata[adata.obs['passed_qc'] == 1],
        ["log1p_n_genes_by_counts"],
        groupby = 'sampleid',
        jitter=0.4,
        size=0.3,
        order=['Control_A','Control_B','Control_C', 'CART_A','CART_B','CART_C','CART_D'],
        palette={"Control_A":"#0e427c", "Control_B":"#5a97c1", "Control_C":"#74c8c3",
                 "CART_A":"#6f1806", "CART_B":"#dd2c09", "CART_C":"#e67424", "CART_D":"#f5c34d"},
        ylabel='log1p(Unique genes expressed)',
        xlabel=None
    )

In [ ]:
with rc_context({"figure.figsize": (7.5, 3), "grid.alpha":0}):
    sc.pl.violin(
        adata[adata.obs['passed_qc'] == 1],
        ["total_counts"],
        groupby = 'sampleid',
        jitter=0.4,
        size=0.3,
        order=['Control_A','Control_B','Control_C', 'CART_A','CART_B','CART_C','CART_D'],
        palette={"Control_A":"#0e427c", "Control_B":"#5a97c1", "Control_C":"#74c8c3",
                 "CART_A":"#6f1806", "CART_B":"#dd2c09", "CART_C":"#e67424", "CART_D":"#f5c34d"},
        ylabel='UMIs',
        xlabel=None
    )

In [ ]:
with rc_context({"figure.figsize": (7.5, 3), "grid.alpha":0}):
    sc.pl.violin(
        adata[adata.obs['passed_qc'] == 1],
        ["log1p_total_counts"],
        groupby = 'sampleid',
        jitter=0.4,
        size=0.3,
        order=['Control_A','Control_B','Control_C', 'CART_A','CART_B','CART_C','CART_D'],
        palette={"Control_A":"#0e427c", "Control_B":"#5a97c1", "Control_C":"#74c8c3",
                 "CART_A":"#6f1806", "CART_B":"#dd2c09", "CART_C":"#e67424", "CART_D":"#f5c34d"},
        ylabel='log1p(UMIs)',
        xlabel=None
    )

In [ ]:
with rc_context({"figure.figsize": (7.5, 3), "grid.alpha":0}):
    sc.pl.violin(
        adata[adata.obs['passed_qc'] == 1],
        ["pct_counts_mt"],
        groupby = 'sampleid',
        jitter=0.4,
        size=0.3,
        order=['Control_A','Control_B','Control_C', 'CART_A','CART_B','CART_C','CART_D'],
        palette={"Control_A":"#0e427c", "Control_B":"#5a97c1", "Control_C":"#74c8c3",
                 "CART_A":"#6f1806", "CART_B":"#dd2c09", "CART_C":"#e67424", "CART_D":"#f5c34d"},
        ylabel='% mitochondrial UMIs',
        xlabel=None
    )

In [ ]:
with rc_context({"figure.figsize": (5, 4), "grid.alpha":0}):
    sc.pl.scatter(adata[adata.obs['passed_qc'] == 1], "log1p_total_counts", "log1p_n_genes_by_counts", 
                  color="pct_counts_mt",
                 title='')

In [ ]:
with rc_context({"figure.figsize": (5, 4), "grid.alpha":0}):
    sc.pl.scatter(adata[adata.obs['passed_qc'] == 1], "log1p_total_counts", "log1p_n_genes_by_counts", 
                  color="sample",
                 title='')

In [ ]:
# filter anndata object
adata = adata[adata.obs['passed_qc'] == 1]

In [ ]:
adata

In [ ]:
# check cell counts by metadata categories post-filtering
adata.obs['sampleid'].value_counts()

In [ ]:
adata.obs['condition'].value_counts()

In [ ]:
pd.crosstab(adata.obs['condition'], adata.obs['sampleid'])

In [ ]:
pd.crosstab(adata.obs['condition'], adata.obs['batch'])

In [ ]:
# adding additional lower bound filter to ensure we're looking at cells with reasonable complexity
sc.pp.filter_cells(adata, min_counts=1000)
sc.pp.filter_cells(adata, min_genes=500)

In [ ]:
# final QC plots

In [ ]:
# plotting QC metrics
with rc_context({"figure.figsize": (5, 3), "grid.alpha":0}):
    sc.pl.violin(
        adata,
        ["n_genes_by_counts", "total_counts", "pct_counts_mt"],
        jitter=0.4,
        size=0.3,
        multi_panel=True
    )

In [ ]:
with rc_context({"figure.figsize": (7.5, 3), "grid.alpha":0}):
    # plot QC metric
    sc.pl.violin(
        adata,
        ["n_genes_by_counts"],
        groupby = 'sampleid',
        jitter=0.4,
        size=0.3,
        order=['Control_A','Control_B','Control_C', 'CART_A','CART_B','CART_C','CART_D'],
        palette={"Control_A":"#0e427c", "Control_B":"#5a97c1", "Control_C":"#74c8c3",
                 "CART_A":"#6f1806", "CART_B":"#dd2c09", "CART_C":"#e67424", "CART_D":"#f5c34d"},
        ylabel='Unique genes expressed',
        xlabel=None
    )

In [ ]:
with rc_context({"figure.figsize": (7.5, 3), "grid.alpha":0}):
    sc.pl.violin(
        adata,
        ["log1p_n_genes_by_counts"],
        groupby = 'sampleid',
        jitter=0.4,
        size=0.3,
        order=['Control_A','Control_B','Control_C', 'CART_A','CART_B','CART_C','CART_D'],
        palette={"Control_A":"#0e427c", "Control_B":"#5a97c1", "Control_C":"#74c8c3",
                 "CART_A":"#6f1806", "CART_B":"#dd2c09", "CART_C":"#e67424", "CART_D":"#f5c34d"},
        ylabel='log1p(Unique genes expressed)',
        xlabel=None
    )

In [ ]:
with rc_context({"figure.figsize": (7.5, 3), "grid.alpha":0}):
    sc.pl.violin(
        adata,
        ["total_counts"],
        groupby = 'sampleid',
        jitter=0.4,
        size=0.3,
        order=['Control_A','Control_B','Control_C', 'CART_A','CART_B','CART_C','CART_D'],
        palette={"Control_A":"#0e427c", "Control_B":"#5a97c1", "Control_C":"#74c8c3",
                 "CART_A":"#6f1806", "CART_B":"#dd2c09", "CART_C":"#e67424", "CART_D":"#f5c34d"},
        ylabel='UMIs',
        xlabel=None
    )

In [ ]:
with rc_context({"figure.figsize": (7.5, 3), "grid.alpha":0}):
    sc.pl.violin(
        adata,
        ["log1p_total_counts"],
        groupby = 'sampleid',
        jitter=0.4,
        size=0.3,
        order=['Control_A','Control_B','Control_C', 'CART_A','CART_B','CART_C','CART_D'],
        palette={"Control_A":"#0e427c", "Control_B":"#5a97c1", "Control_C":"#74c8c3",
                 "CART_A":"#6f1806", "CART_B":"#dd2c09", "CART_C":"#e67424", "CART_D":"#f5c34d"},
        ylabel='log1p(UMIs)',
        xlabel=None
    )

In [ ]:
with rc_context({"figure.figsize": (7.5, 3), "grid.alpha":0}):
    sc.pl.violin(
        adata,
        ["pct_counts_mt"],
        groupby = 'sampleid',
        jitter=0.4,
        size=0.3,
        order=['Control_A','Control_B','Control_C', 'CART_A','CART_B','CART_C','CART_D'],
        palette={"Control_A":"#0e427c", "Control_B":"#5a97c1", "Control_C":"#74c8c3",
                 "CART_A":"#6f1806", "CART_B":"#dd2c09", "CART_C":"#e67424", "CART_D":"#f5c34d"},
        ylabel='% mitochondrial UMIs',
        xlabel=None
    )

In [ ]:
with rc_context({"figure.figsize": (5, 4), "grid.alpha":0}):
    sc.pl.scatter(adata, "log1p_total_counts", "log1p_n_genes_by_counts", color="sampleid",
                 title='')

In [ ]:
# check final post-filtering cell counts by metadata categories
adata.obs['sampleid'].value_counts()

In [ ]:
adata.obs['condition'].value_counts()

In [ ]:
pd.crosstab(adata.obs['condition'], adata.obs['sampleid'])

In [ ]:
pd.crosstab(adata.obs['condition'], adata.obs['batch'])

In [ ]:
# save post-QC-filtering anndata object
adata.write('../output/human/human_adata_object_filtered.h5ad')